# World Bank Health Data Mart — Exploratory Analysis

An interactive analysis of mortality and life expectancy indicators drawn from the **HEALTHDB** Kimball star schema, built on World Bank open data.

## About the Dataset

The **HEALTHDB.DW** data mart contains 28 mortality and life expectancy indicators published by the World Bank, spanning **1960–2024** across **222 geographies** (countries and country groups).

The data is organised as a Kimball star schema:
- **FACT_HEALTH_MEASURES** — one row per country per year, with each indicator as a separate column
- **DIM_DATE** — year-grain calendar dimension
- **DIM_GEOGRAPHY** — country/region dimension with ISO codes

In [ ]:
%%sql -r table_counts
SELECT
    (SELECT COUNT(*) FROM HEALTHDB.DW.FACT_HEALTH_MEASURES) AS FACT_ROWS,
    (SELECT COUNT(*) FROM HEALTHDB.DW.DIM_DATE) AS DATE_ROWS,
    (SELECT COUNT(*) FROM HEALTHDB.DW.DIM_GEOGRAPHY) AS GEO_ROWS

In [ ]:
%%sql -r fact_columns
SELECT
    d.COLUMN_NAME,
    d.DATA_TYPE,
    d.COMMENT
FROM HEALTHDB.INFORMATION_SCHEMA.COLUMNS d
WHERE d.TABLE_SCHEMA = 'DW'
  AND d.TABLE_NAME   = 'FACT_HEALTH_MEASURES'
ORDER BY d.ORDINAL_POSITION

## Measure Categories

| Category | Measures |
|---|---|
| **Life Expectancy** | Total, Female, Male |
| **Maternal Mortality** | Modeled estimate, National estimate |
| **Child & Infant** | Under-5 (T/F/M), Infant (T/F/M), Neonatal |
| **Adult** | Female, Male |
| **NCD (ages 30-70)** | Total, Female, Male |
| **Air Pollution** | Total, Female, Male |
| **Environmental/External** | Unsafe water, Road traffic, Poisoning (T/F/M) |
| **Suicide** | Total, Female, Male |

In [ ]:
%%sql -r data_completeness
SELECT
    MIN(d.YEAR) AS MIN_YEAR,
    MAX(d.YEAR) AS MAX_YEAR,
    COUNT(DISTINCT d.YEAR) AS YEARS_COVERED,
    COUNT(DISTINCT f.GEO_KEY) AS GEOGRAPHIES,
    COUNT(*) AS TOTAL_FACT_ROWS,
    ROUND(100.0 * COUNT(f.LIFE_EXPECTANCY_TOTAL) / COUNT(*), 1) AS PCT_LIFE_EXP_POPULATED,
    ROUND(100.0 * COUNT(f.MORTALITY_INFANT_TOTAL) / COUNT(*), 1) AS PCT_INFANT_MORT_POPULATED,
    ROUND(100.0 * COUNT(f.MATERNAL_MORTALITY_MODELED) / COUNT(*), 1) AS PCT_MATERNAL_MORT_POPULATED,
    ROUND(100.0 * COUNT(f.MORTALITY_SUICIDE_TOTAL) / COUNT(*), 1) AS PCT_SUICIDE_MORT_POPULATED
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

## Global Life Expectancy Trends 🌏

In [ ]:
%%sql -r life_exp_trends
SELECT
    d.YEAR,
    ROUND(AVG(f.LIFE_EXPECTANCY_TOTAL), 2) AS AVG_LIFE_EXP,
    ROUND(AVG(f.LIFE_EXPECTANCY_FEMALE), 2) AS AVG_LIFE_EXP_F,
    ROUND(AVG(f.LIFE_EXPECTANCY_MALE), 2) AS AVG_LIFE_EXP_M
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
WHERE g.LEVEL = 'Country'
  AND f.LIFE_EXPECTANCY_TOTAL IS NOT NULL
GROUP BY d.YEAR
ORDER BY d.YEAR

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(life_exp_trends['YEAR'], life_exp_trends['AVG_LIFE_EXP'],   label='Total', linewidth=2)
ax.plot(life_exp_trends['YEAR'], life_exp_trends['AVG_LIFE_EXP_F'], label='Female', linewidth=1.5, linestyle='--')
ax.plot(life_exp_trends['YEAR'], life_exp_trends['AVG_LIFE_EXP_M'], label='Male',   linewidth=1.5, linestyle='--')
ax.set_title('Global Average Life Expectancy at Birth (Country-level)', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Years')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.show()

## 👶 Child Mortality Trends 👶

In [ ]:
%%sql -r child_mort_trends
SELECT
    d.YEAR,
    ROUND(AVG(f.MORTALITY_UNDER5_TOTAL), 2)  AS AVG_UNDER5,
    ROUND(AVG(f.MORTALITY_INFANT_TOTAL), 2)  AS AVG_INFANT,
    ROUND(AVG(f.MORTALITY_NEONATAL), 2)      AS AVG_NEONATAL
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
WHERE g.LEVEL = 'Country'
  AND f.MORTALITY_UNDER5_TOTAL IS NOT NULL
GROUP BY d.YEAR
ORDER BY d.YEAR

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(child_mort_trends['YEAR'], child_mort_trends['AVG_UNDER5'],  label='Under-5', linewidth=2)
ax.plot(child_mort_trends['YEAR'], child_mort_trends['AVG_INFANT'],  label='Infant',  linewidth=1.5, linestyle='--')
ax.plot(child_mort_trends['YEAR'], child_mort_trends['AVG_NEONATAL'],label='Neonatal', linewidth=1.5, linestyle=':')
ax.set_title('Global Average Child Mortality Rates (per 1,000 live births)', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Deaths per 1,000')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.show()

## 🗺️ Country Comparisons 🗺️

In [ ]:
%%sql -r top_countries
SELECT
    g.GEO_NAME,
    g.ISO_ALPHA3,
    ROUND(AVG(f.LIFE_EXPECTANCY_TOTAL), 1) AS AVG_LIFE_EXP,
    ROUND(AVG(f.MORTALITY_INFANT_TOTAL), 1) AS AVG_INFANT_MORT,
    ROUND(AVG(f.MORTALITY_UNDER5_TOTAL), 1) AS AVG_UNDER5_MORT
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
WHERE g.LEVEL = 'Country'
  AND d.YEAR >= 2015
  AND f.LIFE_EXPECTANCY_TOTAL IS NOT NULL
GROUP BY g.GEO_NAME, g.ISO_ALPHA3
ORDER BY AVG_LIFE_EXP DESC
LIMIT 20

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
y_pos = range(len(top_countries))
ax.barh(y_pos, top_countries['AVG_LIFE_EXP'], color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_countries['GEO_NAME'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Average Life Expectancy (years)')
ax.set_title('Top 20 Countries by Life Expectancy (2015–2024)', fontsize=14)
ax.set_xlim(left=70)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 🤰 Maternal Mortality Trends🤰

In [ ]:
%%sql -r maternal_mort
SELECT
    d.YEAR,
    ROUND(AVG(f.MATERNAL_MORTALITY_MODELED), 1) AS AVG_MATERNAL_MORT_MODELED,
    COUNT(f.MATERNAL_MORTALITY_MODELED) AS COUNTRIES_REPORTING
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
WHERE g.LEVEL = 'Country'
  AND f.MATERNAL_MORTALITY_MODELED IS NOT NULL
GROUP BY d.YEAR
ORDER BY d.YEAR

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(maternal_mort['YEAR'], maternal_mort['AVG_MATERNAL_MORT_MODELED'], color='crimson', linewidth=2)
ax1.set_title('Global Average Maternal Mortality Ratio (modeled, per 100k live births)', fontsize=14)
ax1.set_xlabel('Year')
ax1.set_ylabel('Deaths per 100,000 live births', color='crimson')
ax1.tick_params(axis='y', labelcolor='crimson')
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(mticker.MultipleLocator(5))
ax2 = ax1.twinx()
ax2.bar(maternal_mort['YEAR'], maternal_mort['COUNTRIES_REPORTING'], alpha=0.15, color='grey', label='Countries reporting')
ax2.set_ylabel('Countries Reporting', color='grey')
ax2.tick_params(axis='y', labelcolor='grey')
plt.tight_layout()
plt.show()

## Statistical Analysis 📊

In [ ]:
%%sql -r corr_data
SELECT
    g.GEO_NAME,
    f.LIFE_EXPECTANCY_TOTAL,
    f.MORTALITY_INFANT_TOTAL
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
WHERE g.LEVEL = 'Country'
  AND d.YEAR = 2020
  AND f.LIFE_EXPECTANCY_TOTAL IS NOT NULL
  AND f.MORTALITY_INFANT_TOTAL IS NOT NULL

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(corr_data['MORTALITY_INFANT_TOTAL'], corr_data['LIFE_EXPECTANCY_TOTAL'], alpha=0.6, edgecolors='w', s=50)
z = np.polyfit(corr_data['MORTALITY_INFANT_TOTAL'], corr_data['LIFE_EXPECTANCY_TOTAL'], 1)
p = np.poly1d(z)
x_line = np.linspace(corr_data['MORTALITY_INFANT_TOTAL'].min(), corr_data['MORTALITY_INFANT_TOTAL'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.7)
r = corr_data['MORTALITY_INFANT_TOTAL'].corr(corr_data['LIFE_EXPECTANCY_TOTAL'])
ax.set_title(f'Life Expectancy vs Infant Mortality (2020)  —  r = {r:.3f}', fontsize=14)
ax.set_xlabel('Infant Mortality (per 1,000 live births)')
ax.set_ylabel('Life Expectancy (years)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
%%sql -r life_exp_dist
SELECT
    g.GEO_NAME,
    f.LIFE_EXPECTANCY_TOTAL
FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
WHERE g.LEVEL = 'Country'
  AND d.YEAR = 2020
  AND f.LIFE_EXPECTANCY_TOTAL IS NOT NULL

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(life_exp_dist['LIFE_EXPECTANCY_TOTAL'], bins=25, color='steelblue', edgecolor='white')
med = life_exp_dist['LIFE_EXPECTANCY_TOTAL'].median()
ax.axvline(med, color='red', linestyle='--', label=f'Median: {med:.1f}')
ax.set_title('Distribution of Life Expectancy Across Countries (2020)', fontsize=14)
ax.set_xlabel('Life Expectancy (years)')
ax.set_ylabel('Number of Countries')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
%%sql -r yoy_changes
WITH lagged AS (
    SELECT
        f.GEO_KEY,
        g.GEO_NAME,
        d.YEAR,
        f.LIFE_EXPECTANCY_TOTAL,
        LAG(f.LIFE_EXPECTANCY_TOTAL) OVER (PARTITION BY f.GEO_KEY ORDER BY d.YEAR) AS PREV_LIFE_EXP
    FROM HEALTHDB.DW.FACT_HEALTH_MEASURES f
    JOIN HEALTHDB.DW.DIM_DATE d ON f.DATE_KEY = d.DATE_KEY
    JOIN HEALTHDB.DW.DIM_GEOGRAPHY g ON f.GEO_KEY = g.GEO_KEY
    WHERE g.LEVEL = 'Country'
      AND f.LIFE_EXPECTANCY_TOTAL IS NOT NULL
)
SELECT
    YEAR,
    ROUND(AVG(LIFE_EXPECTANCY_TOTAL - PREV_LIFE_EXP), 3) AS AVG_YOY_CHANGE,
    ROUND(MIN(LIFE_EXPECTANCY_TOTAL - PREV_LIFE_EXP), 3) AS MIN_YOY_CHANGE,
    ROUND(MAX(LIFE_EXPECTANCY_TOTAL - PREV_LIFE_EXP), 3) AS MAX_YOY_CHANGE
FROM lagged
WHERE PREV_LIFE_EXP IS NOT NULL
GROUP BY YEAR
ORDER BY YEAR

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if v >= 0 else 'red' for v in yoy_changes['AVG_YOY_CHANGE']]
ax.bar(yoy_changes['YEAR'], yoy_changes['AVG_YOY_CHANGE'], color=colors, alpha=0.7)
ax.fill_between(yoy_changes['YEAR'], yoy_changes['MIN_YOY_CHANGE'], yoy_changes['MAX_YOY_CHANGE'], alpha=0.1, color='grey', label='Min–Max range')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Year-over-Year Change in Life Expectancy (Country Average)', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Change (years)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Life expectancy has steadily risen** globally since 1960, with a visible dip around 2020 corresponding to the COVID-19 pandemic.
2. **Female life expectancy consistently exceeds male** by approximately 4–5 years across the full time series.
3. **Child mortality has declined dramatically** — under-5 mortality has fallen from ~200 to ~40 per 1,000 live births over six decades.
4. **Strong negative correlation** between infant mortality and life expectancy (r ≈ -0.9) confirms that child survival is a key driver of overall longevity.
5. **Maternal mortality remains high** in many regions despite a global downward trend, with wide disparity between the highest and lowest countries.
6. **Year-over-year analysis reveals** that most years show positive gains, but certain shock events (pandemics, conflicts) cause visible reversals.

## Next Steps

- **Regional drill-downs**: Aggregate by continent or income group to identify where interventions are most needed.
- **Predictive modelling**: Forecast life expectancy and child mortality rates using time-series models (e.g., ARIMA, Prophet).
- **Cause-of-death analysis**: Leverage the NCD, air pollution, road traffic, and suicide mortality columns for deeper cause-specific insights.
- **Gender gap study**: Quantify how the male–female mortality gap varies by region and over time.
- **Correlation matrix**: Build a full correlation heatmap across all 28 measures to discover unexpected relationships.
- **Dashboard**: Publish key visuals as a Streamlit in Snowflake app for interactive stakeholder exploration.